<h>패키지</h>

In [117]:
import pandas as pd
import numpy as np
import requests
import time

# 시각화 패키지
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 경고창 무시
import warnings
warnings.filterwarnings("ignore")

<h>Riot Account ID 정보 불러오기</h>

In [118]:
# API 키 갱신 확인 후 사용
api_key = "RGAPI-d9685e24-88d7-457b-bb97-f6dac20b61a1"

# 소환사 이름
gameName = "슈 닷"
tagLine = "77777"
acount_url = f"https://asia.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}"
account_r = requests.get(summoner_url)

account_r

# 나의 accountId
puuid = summoner_r.json()['puuid']

<h>Find Matches by PUUID</h>

In [119]:
beginIndex = 0

# 매치 정보 (지수표현으로 나오지 않는지 확인)
my_matchids = np.array([], dtype=int)

while True:
    match_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start={beginIndex}&count=20&api_key={api_key}"
    match_r = requests.get(match_url)
    
    # stop 조건: 불러온 json 파일의 length
    #if len(pd.json_normalize(match_r.json())) == 0:
        #break
    
    # 테스트로 100개의 데이터를 불러오고 스톱
    if beginIndex > 100:
        break
    
    # 한번에 20의 매치 정보
    temp_matchids = match_r.json()
    
    # gameid stack
    my_matchids = np.concatenate([my_matchids, temp_matchids])
    
    # start index 변경
    beginIndex += 20
    
    # RATE LIMITS 피하기
    time.sleep(1)

<p>Match Info 불러오기</p>

In [120]:
my_game = pd.DataFrame()

for i in my_matchids:
    matchid = i

    game_url = f"https://asia.api.riotgames.com/lol/match/v5/matches/{matchid}?api_key={api_key}"
    game_r = requests.get(game_url)

    # 10명의 대전 정보
    temp_game = pd.json_normalize(game_r.json()['info']['participants'])
    
    # 나의 participantid 확인
    summoner_lst = pd.json_normalize(game_r.json()['info']['participants'])
    me = summoner_lst[summoner_lst['puuid'] == puuid]
    
    # 나의 대전 정보
    temp_game = temp_game[temp_game['participantId'] == int(me['participantId'])]
    
    # 추가 정보 (게임 종류, 게임 시간)
    temp_game['queueId'] = game_r.json()['info']['queueId']
    temp_game['gameDuration'] = game_r.json()['info']['gameDuration']

    # 대전 정보 stack
    my_game = pd.concat([my_game, temp_game])
    
    # RATE LIMITS 피하기
    time.sleep(1)

# 시간이 오래걸리므로 csv로 저장
my_game.to_csv(f"{gameName}_log.csv", index=False)

<p>데이터 전처리</p>

In [126]:
my_games = pd.read_csv('슈 닷_log.csv')
my_games.head()

# 특정 컬럼만 사용
col_lst = ["queueId", "teamId", "gameDuration", "championName", 
            "win", "kills", "deaths", "assists", 
            "lane", "item0", "item1", "item2", "item3", "item4", "item5", "item6",
            "perks.statPerks.defense", "perks.statPerks.flex", "perks.statPerks.offense", 
            "perks.styles", "challenges.hadAfkTeammate"]

my_games2 = my_games[col_lst]

# 랭크(420)만 사용
my_games2 = my_games2[my_games2["queueId"].isin([420])]

# 게임 시간 4분 미만 제거(다시하기 제거하는 용도)
my_games2[my_games2["gameDuration"] < 240]

my_games2.head(50)

,queueId,teamId,gameDuration,championName,win,kills,deaths,assists,lane,item0,...,item2,item3,item4,item5,item6,perks.statPerks.defense,perks.statPerks.flex,perks.statPerks.offense,perks.styles,challenges.hadAfkTeammate
7,420,100,1842,Ezreal,True,1,8,8,BOTTOM,1055,...,3158,3042,1036,1036,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
8,420,100,947,Kaisa,False,0,4,0,NONE,1055,...,3006,1043,6670,0,3340,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
27,420,100,2126,Nilah,False,10,7,8,BOTTOM,3026,...,6675,3036,3072,3006,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
34,420,100,1337,Ziggs,True,2,3,2,BOTTOM,6653,...,1052,0,3020,1056,3340,5002,5008,5008,"[{'description': 'primaryStyle', 'selections':...",NaN
35,420,200,2295,Jinx,False,5,7,14,BOTTOM,3026,...,3006,3095,3036,1018,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
36,420,200,1346,Kalista,True,8,4,10,BOTTOM,1055,...,3051,3006,1043,3124,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
37,420,200,1460,Aphelios,False,2,6,5,BOTTOM,1055,...,3006,6672,0,0,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
38,420,100,1581,Varus,False,4,7,1,BOTTOM,3153,...,1053,3006,1018,1055,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
42,420,200,1936,MissFortune,False,5,4,3,BOTTOM,1055,...,3009,3814,3042,3035,3363,5002,5008,5008,"[{'description': 'primaryStyle', 'selections':...",NaN
43,420,100,1614,Jinx,False,3,5,1,BOTTOM,1055,...,3095,3006,3086,1036,3363,5002,5008,5005,"[{'description': 'primaryStyle', 'selections':...",NaN
